# Team Challenge · Sprint 03-04 - Catálogo de películas

**Dataset:**  
- [movies.csv](./data/movies.csv)
- [ratings.csv](./data/ratings.csv)
- [tags.csv](./data/tags.csv)
- [links.csv](./data/links.csv) 

**Entregables:** Repositorio de GitHub con el código fuente. Puede ser scripts de python o notebook ordenado, código reproducible, README con cómo ejecutar el proyecto.

**Control de versiones:** gestión del proyecto con **GitHub desde el primer día**. Trabajar en equipo en paralelo con **ramas**, **pull requests** y revisión entre compañeros.

**Parte 1 (Pandas):** Data analytics (Pandas)

**Parte 2 (TMDB):** una petición `GET /3/movie/{id}` por película; definid `TMDB_API_KEY` o `TMDB_READ_ACCESS_TOKEN` en el entorno (sin subir claves al repo).

**Parte 3 (Gemini):** añadir `overview_es` a `movies10` a partir de los `overview` de TMDB; definid `GEMINI_API_KEY` o usad `getpass` (sin subir claves al repo). Requiere `pip install google-genai`.

**Parte 4 (opcional):**
- **A)** catálogo ampliado (`pitch_es`, `edad_sugerida`, `temas`).
- **B)** recomendador por género.

## Trabajo en equipo y control de versiones

- Crear el **repositorio en GitHub** al inicio (día 1–2), no al final.
- Definir una rama principal (`main`) y una para desarrollo (`develop`) y ramas por tarea (p. ej. `feature/parte1-pandas`, `feature/tmdb`, `feature/gemini`).
- Integrar el trabajo con **pull requests** hacia `develop`; al menos **una PR revisada y mergeada por miembro** del equipo.
- Evitar trabajar en en `main` es la rama de "producción"
- Resolver conflictos en ramas antes del merge.
- **README:** cómo clonar el repo, instalar dependencias, configurar claves (`TMDB_*`, `GEMINI_API_KEY`) y ejecutar el notebook o scripts.
- **No subir claves** al repositorio (usar `.gitignore` para `.env` si aplica).

## Parte 1: Data analytics (Pandas)

### 1. Ingesta de datos

- Los CSV están en **`Team_Challenges/TC_01_Sprint_03_04/data/`**.
- Cargar con **Pandas** los cuatro ficheros: `movies.csv`, `ratings.csv`, `tags.csv`, `links.csv`.
- Comprobar para cada `DataFrame`: `shape`, columnas, `dtypes`, `head` y conteo de nulos.

In [7]:
# --- Apartado 1: Ingesta de datos ---
## Importación de librerías
import pandas as pd

## Carga de datasets

In [8]:
movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")
tags = pd.read_csv("../data/tags.csv")
links = pd.read_csv("../data/links.csv")

## Exploración inicial

In [9]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [11]:
datasets = {
    "Movies": movies,
    "Ratings": ratings,
    "Tags": tags,
    "Links": links
}

for nombre, df in datasets.items():
    print("=" * 60)
    print(nombre)
    print("=" * 60)

    print("\nShape:")
    print(df.shape)

    print("\nColumnas:")
    print(df.columns.tolist())

    print("\nTipos de datos:")
    print(df.dtypes)

    print("\nPrimeras filas:")
    display(df.head())

    print("\nValores nulos:")
    print(df.isnull().sum())

    print("\n")

Movies

Shape:
(9742, 3)

Columnas:
['movieId', 'title', 'genres']

Tipos de datos:
movieId    int64
title        str
genres       str
dtype: object

Primeras filas:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Valores nulos:
movieId    0
title      0
genres     0
dtype: int64


Ratings

Shape:
(100836, 4)

Columnas:
['userId', 'movieId', 'rating', 'timestamp']

Tipos de datos:
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

Primeras filas:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931



Valores nulos:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


Tags

Shape:
(3683, 4)

Columnas:
['userId', 'movieId', 'tag', 'timestamp']

Tipos de datos:
userId       int64
movieId      int64
tag            str
timestamp    int64
dtype: object

Primeras filas:


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200



Valores nulos:
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64


Links

Shape:
(9742, 3)

Columnas:
['movieId', 'imdbId', 'tmdbId']

Tipos de datos:
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object

Primeras filas:


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0



Valores nulos:
movieId    0
imdbId     0
tmdbId     8
dtype: int64




### 2. Columna `year` desde el título

- En el dataset de películas. el año de estreno suele aparecer **entre paréntesis al final** de `title`, p. ej. `Batman (1989)`.
- Implementad una función (`year_from_title`) que devuelva un entero de cuatro cifras o valor ausente (`NaN` / `<NA>`) si el título no sigue ese patrón.
- Añadid la columna **`year`** a `movies` como numérico (`pd.to_numeric(..., errors="coerce")`).
- Editar el campo `title` para que no contenga el año de estreno.
- Contad cuántas películas **no** tienen año reconocible y mostrad **una pequeña muestra** de sus títulos (casos límite).

In [13]:
# --- Apartado 2: columna `year` ---

import re
import pandas as pd

def year_from_title(title):
    """
    Extrae el año de estreno si aparece al final del título
    entre paréntesis. Si no existe devuelve pd.NA.
    """
    match = re.search(r"\((\d{4})\)$", title)

    if match:
        return int(match.group(1))

    return pd.NA

In [14]:
movies["year"] = movies["title"].apply(year_from_title)

In [15]:
movies["year"] = pd.to_numeric(
    movies["year"],
    errors="coerce"
)

In [16]:
movies["title"] = movies["title"].str.replace(
    r"\s*\(\d{4}\)$",
    "",
    regex=True
)

In [17]:
missing_years = movies["year"].isna().sum()

print(f"Películas sin año reconocido: {missing_years}")

Películas sin año reconocido: 24


In [18]:
movies.loc[
    movies["year"].isna(),
    ["title"]
].head(10)

,title
5609,From Dusk Till Dawn 2: Texas Blood Money (1999)
6059,Babylon 5
6706,Justice League: The New Frontier (2008)
6718,Assembly (Ji jie hao) (2007)
7878,96 Minutes (2011)
7896,Superman/Doomsday (2007)
7910,Pocahontas II: Journey to a New World (1998)
7978,Runaway Brain (1995)
8148,Justice League: Doom (2012)
8228,3 dev adam (Three Giant Men) (1973)


In [19]:
movies.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji,Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men,Comedy|Romance,1995.0
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II,Comedy,1995.0


### 2. Unificación de datos (*merge* / *join*)

- Entender las **claves** entre tablas (p. ej. `movieId` une `movies`, `ratings`, `tags` y `links`).
- Construir un esquema unificado: al menos una tabla **película–usuario–rating** (ratings enriquecida con título y géneros) y otra con **película–tags** si procede.
- Usar `merge` (u operaciones equivalentes) con criterio claro: tipo de unión (*inner* / *left*), duplicados generados y cómo se resuelven.
- Imprimir las 10 primeras filas de las tablas resultantes.
- Dejar documentado qué filas se pierden o se multiplican al unir y **por qué** es aceptable en vuestro caso de uso.

In [ ]:
# --- Apartado 3: Merge / join  ---

Las cuatro tablas del proyecto están relacionadas mediante la columna **movieId**:

- **movies**: información de las películas (título, géneros y año).
- **ratings**: valoraciones realizadas por los usuarios.
- **tags**: etiquetas asignadas por los usuarios.
- **links**: identificadores de IMDb y TMDB.

La clave común entre todas ellas es **movieId**, que utilizaremos para realizar las uniones mediante `merge()`.

In [21]:
# Unimos ratings con la información de las películas
ratings_movies = pd.merge(
    ratings,
    movies,
    on="movieId",
    how="left"
)

print("Primeras 10 filas de la tabla película–usuario–rating:")
ratings_movies.head(10)

Primeras 10 filas de la tabla película–usuario–rating:


,userId,movieId,rating,timestamp,title,genres,year
0,1,1,4.0,964982703,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,1,3,4.0,964981247,Grumpier Old Men,Comedy|Romance,1995.0
2,1,6,4.0,964982224,Heat,Action|Crime|Thriller,1995.0
3,1,47,5.0,964983815,Seven (a.k.a. Se7en),Mystery|Thriller,1995.0
4,1,50,5.0,964982931,"Usual Suspects, The",Crime|Mystery|Thriller,1995.0
5,1,70,3.0,964982400,From Dusk Till Dawn,Action|Comedy|Horror|Thriller,1996.0
6,1,101,5.0,964980868,Bottle Rocket,Adventure|Comedy|Crime|Romance,1996.0
7,1,110,4.0,964982176,Braveheart,Action|Drama|War,1995.0
8,1,151,5.0,964984041,Rob Roy,Action|Drama|Romance|War,1995.0
9,1,157,5.0,964984100,Canadian Bacon,Comedy|War,1995.0


In [22]:
# Unimos tags con la información de las películas
tags_movies = pd.merge(
    tags,
    movies,
    on="movieId",
    how="left"
)

print("Primeras 10 filas de la tabla película–tags:")
tags_movies.head(10)

Primeras 10 filas de la tabla película–tags:


,userId,movieId,tag,timestamp,title,genres,year
0,2,60756,funny,1445714994,Step Brothers,Comedy,2008.0
1,2,60756,Highly quotable,1445714996,Step Brothers,Comedy,2008.0
2,2,60756,will ferrell,1445714992,Step Brothers,Comedy,2008.0
3,2,89774,Boxing story,1445715207,Warrior,Drama,2011.0
4,2,89774,MMA,1445715200,Warrior,Drama,2011.0
5,2,89774,Tom Hardy,1445715205,Warrior,Drama,2011.0
6,2,106782,drugs,1445715054,"Wolf of Wall Street, The",Comedy|Crime|Drama,2013.0
7,2,106782,Leonardo DiCaprio,1445715051,"Wolf of Wall Street, The",Comedy|Crime|Drama,2013.0
8,2,106782,Martin Scorsese,1445715056,"Wolf of Wall Street, The",Comedy|Crime|Drama,2013.0
9,7,48516,way too long,1169687325,"Departed, The",Crime|Drama|Thriller,2006.0


In [23]:
print("Número de filas originales y tras el merge:\n")

print(f"ratings: {ratings.shape[0]} filas")
print(f"ratings_movies: {ratings_movies.shape[0]} filas\n")

print(f"tags: {tags.shape[0]} filas")
print(f"tags_movies: {tags_movies.shape[0]} filas")

Número de filas originales y tras el merge:

ratings: 100836 filas
ratings_movies: 100836 filas

tags: 3683 filas
tags_movies: 3683 filas


In [24]:
print("Valores nulos tras el merge:\n")

print(ratings_movies[["title", "genres"]].isna().sum())

print()

print(tags_movies[["title", "genres"]].isna().sum())

Valores nulos tras el merge:

title     0
genres    0
dtype: int64

title     0
genres    0
dtype: int64


### Tipo de unión utilizado

Se ha utilizado un **LEFT JOIN** (`how="left"`).

Este tipo de unión conserva todas las filas de `ratings` y `tags`, añadiendo la información de las películas desde `movies`.

Es la opción más adecuada porque el objetivo es mantener todas las valoraciones y todas las etiquetas realizadas por los usuarios.

### Filas perdidas y filas multiplicadas

No se pierden filas al realizar la unión, ya que el `LEFT JOIN` conserva todas las filas de la tabla izquierda (`ratings` o `tags`).

Sin embargo, algunas películas aparecen repetidas. Esto ocurre porque la relación entre `movies` y `ratings` es de **uno a muchos**: una película puede recibir múltiples valoraciones de distintos usuarios.

Del mismo modo, una película puede tener varias etiquetas asignadas por diferentes usuarios, por lo que también se repite en la tabla `tags_movies`.

Estas repeticiones son esperadas y no deben eliminarse, ya que cada fila representa una interacción diferente entre un usuario y una película.

### 4. Agregaciones y segmentación

- Usar `groupby` (por usuario, por película o por género) con al menos una agregación multi-columna (`agg`).
- Comparar dos segmentos (p. ej. usuarios con muchas valoraciones vs. pocos; o por **década** usando la columna `year` del apartado 3) con tablas resumen.

In [34]:
# --- Apartado 4: Agregaciones ---


En este apartado se utilizará `groupby()` para obtener estadísticas resumidas del conjunto de datos.

Se realizará:

- Una agregación por usuario utilizando varias funciones mediante `agg()`.
- Una segmentación de usuarios según el número de valoraciones realizadas.
- Una comparación entre ambos segmentos mediante tablas resumen.

In [25]:
# Estadísticas por usuario
user_stats = ratings_movies.groupby("userId").agg(
    total_ratings=("rating", "count"),
    average_rating=("rating", "mean"),
    min_rating=("rating", "min"),
    max_rating=("rating", "max")
)

print("Primeras 10 filas:")
user_stats.head(10)

Primeras 10 filas:


,total_ratings,average_rating,min_rating,max_rating
userId,,,,
1,232,4.366379,1.0,5.0
2,29,3.948276,2.0,5.0
3,39,2.435897,0.5,5.0
4,216,3.555556,1.0,5.0
5,44,3.636364,1.0,5.0
6,314,3.493631,1.0,5.0
7,152,3.230263,0.5,5.0
8,47,3.574468,1.0,5.0
9,46,3.260870,1.0,5.0


In [26]:
user_stats["segment"] = user_stats["total_ratings"].apply(
    lambda x: "Muchas valoraciones" if x >= 50 else "Pocas valoraciones"
)

user_stats.head()

,total_ratings,average_rating,min_rating,max_rating,segment
userId,,,,,
1,232,4.366379,1.0,5.0,Muchas valoraciones
2,29,3.948276,2.0,5.0,Pocas valoraciones
3,39,2.435897,0.5,5.0,Pocas valoraciones
4,216,3.555556,1.0,5.0,Muchas valoraciones
5,44,3.636364,1.0,5.0,Pocas valoraciones


In [27]:
segment_summary = user_stats.groupby("segment").agg(
    users=("total_ratings", "count"),
    avg_num_ratings=("total_ratings", "mean"),
    avg_rating=("average_rating", "mean")
)

segment_summary

,users,avg_num_ratings,avg_rating
segment,,,
Muchas valoraciones,385,243.667532,3.626051
Pocas valoraciones,225,31.217778,3.710560


### Comparación de segmentos

Se han definido dos grupos de usuarios:

- **Muchas valoraciones:** usuarios con 50 o más valoraciones.
- **Pocas valoraciones:** usuarios con menos de 50 valoraciones.

La tabla resumen permite comparar el número de usuarios de cada grupo, la cantidad media de valoraciones realizadas y la puntuación media otorgada por cada segmento.

### 5. Preguntas sobre los datos (`movies`)

**Requisito:** haber creado la columna `year` en el **apartado 2**.

1. **Cuántas** películas están listadas en `movies`.
2. **Cuáles** son las **más antiguas** (menor año extraído del título).
3. **Cuántas** tienen **"Dracula"** en el título (coincidencia parcial, sin distinguir mayúsculas).
4. **Títulos más comunes**.
5. Películas con **"Exorcist"** ordenadas de la más antigua a la más moderna.
6. **Cuántas** con año **1950**.
7. **Cuántas** entre **1950 y 1959** inclusive.
8. **Año** de la película con título exacto **`Batman`** (y contraste con otras *Batman*).
9. Listado de películas que tienen como tag "sci-fi" y "adventure"
10. ¿Cuál es la tag más repetida?

In [28]:
#1. **Cuántas** películas están listadas en `movies`.
print(f"Número de películas: {movies.shape[0]}")

Número de películas: 9742


In [29]:
#2. **Cuáles** son las **más antiguas** (menor año extraído del título)
oldest_year = movies["year"].min()

movies[movies["year"] == oldest_year][["title", "year"]]

,title,year
5868,"Trip to the Moon, A (Voyage dans la lune, Le)",1902.0


In [30]:
#3. **Cuántas** tienen **"Dracula"** en el título (coincidencia parcial, sin distinguir mayúsculas).
dracula_movies = movies[movies["title"].str.contains("Dracula", case=False, na=False)]

print(f"Películas con 'Dracula' en el título: {len(dracula_movies)}")

dracula_movies[["title", "year"]]

Películas con 'Dracula' en el título: 9


,title,year
11,Dracula: Dead and Loving It,1995.0
1018,Blood for Dracula (Andy Warhol's Dracula),1974.0
1027,Dracula (Bram Stoker's Dracula),1992.0
1990,Dracula,1931.0
3012,Dracula 2000,2000.0
4002,Horror of Dracula (Dracula),1958.0
4503,Dracula,1979.0
8529,Dracula Untold,2014.0
9003,Fracchia contro Dracula,1985.0


In [31]:
#4. **Títulos más comunes**.
title_counts = (
    movies["title"]
    .value_counts()
    .reset_index()
)

title_counts.columns = ["title", "count"]

title_counts.head(10)

,title,count
0,Hamlet,5
1,"Misérables, Les",4
2,"Three Musketeers, The",4
3,Jane Eyre,4
4,"Christmas Carol, A",4
5,Little Women,3
6,"Hunchback of Notre Dame, The",3
7,Emma,3
8,Spellbound,3
9,Cinderella,3


In [32]:
#5. Películas con **"Exorcist"** ordenadas de la más antigua a la más moderna.
exorcist_movies = movies[
    movies["title"].str.contains("Exorcist", case=False, na=False)
].sort_values("year")

exorcist_movies[["title", "year"]]

,title,year
1472,"Exorcist, The",1973.0
1473,Exorcist II: The Heretic,1977.0
1474,"Exorcist III, The",1990.0
5315,Exorcist: The Beginning,2004.0
5904,Dominion: Prequel to the Exorcist,2005.0
9173,Blue Exorcist: The Movie,2012.0


In [33]:
#6. **Cuántas** con año **1950**
movies_1950 = movies[movies["year"] == 1950]

print(f"Películas estrenadas en 1950: {len(movies_1950)}")

Películas estrenadas en 1950: 21


In [34]:
#7. **Cuántas** entre **1950 y 1959** inclusive
movies_50s = movies[
    movies["year"].between(1950, 1959)
]

print(f"Películas entre 1950 y 1959: {len(movies_50s)}")

Películas entre 1950 y 1959: 279


In [35]:
#8. **Año** de la película con título exacto **`Batman`** (y contraste con otras *Batman*).
print("Película con título exacto 'Batman':")

display(
    movies[movies["title"] == "Batman"][["title", "year"]]
)

print("\nOtras películas relacionadas con Batman:")

display(
    movies[
        movies["title"].str.contains("Batman", case=False, na=False)
    ][["title", "year"]].sort_values("year")
)

Película con título exacto 'Batman':


,title,year
509,Batman,1989.0
5463,Batman,1966.0



Otras películas relacionadas con Batman:


,title,year
5463,Batman,1966.0
509,Batman,1989.0
1060,Batman Returns,1992.0
2418,Batman: Mask of the Phantasm,1993.0
126,Batman Forever,1995.0
1174,Batman & Robin,1997.0
5620,"Batman/Superman Movie, The",1998.0
5631,Batman Beyond: Return of the Joker,2000.0
8234,Batman: Mystery of the Batwoman,2003.0
5917,Batman Begins,2005.0


In [36]:
#9. Listado de películas que tienen como tag "sci-fi" y "adventure"
selected_tags = tags[
    tags["tag"].str.lower().isin(["sci-fi", "adventure"])
]

movies_tags = pd.merge(
    selected_tags,
    movies,
    on="movieId",
    how="left"
)

movies_tags[["title", "tag"]].drop_duplicates().sort_values("title")

,title,tag
31,2001: A Space Odyssey,sci-fi
16,Aliens,sci-fi
3,"Animatrix, The",sci-fi
28,Avatar,sci-fi
13,Blade Runner,sci-fi
9,Cowboy Bebop: The Movie (Cowboy Bebop: Tengoku...,sci-fi
4,"Da Vinci Code, The",adventure
30,Final Fantasy: The Spirits Within,sci-fi
7,"Hobbit: The Desolation of Smaug, The",adventure
20,Inception,sci-fi


In [37]:
#10. ¿Cuál es la tag más repetida
most_common_tag = (
    tags["tag"]
    .str.lower()
    .value_counts()
    .reset_index()
)

most_common_tag.columns = ["tag", "count"]

most_common_tag.head(1)

,tag,count
0,in netflix queue,131


## Parte 2: Petición HTTP a la API de TMDB

### Endpoint

`GET https://api.themoviedb.org/3/movie/{tmdb_id}`

Ejemplo público de la misma forma que en la documentación de TMDB: **`https://api.themoviedb.org/3/movie/100`** (el número es el `tmdb_id`; en vuestro caso usaréis el `tmdbId` de cada fila de `links`).

### Tareas

1. Construir un **`DataFrame` `movies10`** con **10 películas** del dataset original (por ejemplo las 10 primeras filas que tengan `tmdbId` tras unir `movies` con `links`).
2. Escribir una función **`fetch_movie_details(tmdb_id)`** que haga la petición anterior y devuelva al menos **`overview`** y **`homepage`** (texto vacío si vienen nulos).
3. Recorrer `movies10` y **añadir** a cada registro esas dos columnas en el propio `DataFrame`.

### Autenticación

Para poder acceder a la API de TMDB, debéis hacer uso del ACCESS TOKEN o de la API KEY que te proporcionan al registrarse. Recomendamos probar los endpoint en POSTMAN para hacer las pruebas de la llamada a la API antes de crear el script en Python. Deberíais tener en vuestro proyecto:**`TMDB_API_KEY`** (query `api_key`) o **`TMDB_READ_ACCESS_TOKEN`** (cabecera `Authorization: Bearer …`). Podéis crear un proyecto con variables de entorno o usar una celda de código usando **`getpass`** (entrada oculta, solo esa sesión del kernel). 

No guardéis claves en el notebook

In [36]:
# --- Parte 2: TMDB GET /movie/{id} → overview y homepage  ---
# Requiere: `movies` y `links` cargados. pip install requests

## Parte 3: Sinopsis en español con Gemini

Usad el `DataFrame` **`movies10`** de la Parte 2 (columna `overview` en inglés desde TMDB).

### Tareas

1. Configurar **`GEMINI_API_KEY`** (variable de entorno o `getpass`, como en Sprint 4).
2. Crear el cliente **`google-genai`** y una función **`summarize_overview_es(overview, title="")`** que devuelva un **resumen en español de máximo 2 frases**. Si `overview` está vacío, devolver cadena vacía **sin llamar a la API**.
3. Recorrer `movies10` y añadir la columna **`overview_es`**.
4. Mostrar `title`, `overview` (recorte) y `overview_es` para **3 películas**.

### Autenticación

Clave en [Google AI Studio](https://aistudio.google.com/). Variable **`GEMINI_API_KEY`** o celda con **`getpass`**. No guardéis claves en el notebook.

### Prerrequisitos

- Parte 2 ejecutada (`movies10` con columna `overview`).
- `pip install google-genai`

In [37]:
# --- Parte 3: overview → overview_es con Gemini  ---
# Requiere: `movies10` de la Parte 2. pip install google-genai

## Parte 4 (opcional): extensiones con Gemini

**No obligatorio.** Solo si el grupo terminó las partes 1-3. Podéis elegir **A**, **B** o ambas. Son independientes.

---

### A) Catálogo ampliado en `movies10`

Partiendo de `movies10` (con `overview` y, si ya lo tenéis, `overview_es`), añadid con **una llamada JSON por película**:

- **`pitch_es`**: texto de cartelera en español (máx. 280 caracteres).
- **`edad_sugerida`**: uno de `TP`, `+7`, `+12`, `+16`, `+18`.
- **`temas`**: exactamente 3 temas en español (en el DataFrame, cadena separada por comas).

Función sugerida: **`enrich_catalog_fields(overview, title="", genres="", year=None)`**. Basad la respuesta solo en sinopsis y metadatos; no inventéis reparto ni datos externos. Si `overview` está vacío, no llaméis a la API.

Mostrad 2–3 filas con las columnas nuevas.

---

### B) Recomendador por género

Usad **`ratings_movies`** (Parte 1) y Gemini.

1. **`top_rated_by_genre(ratings_movies, genres, top_n=10, min_ratings=50)`** — filtra por género(s), nota media por `movieId`, devuelve el top N (mínimo `min_ratings` valoraciones por película).
2. **`recommend_movies(candidates, favorite_genres, n=3)`** — el modelo recomienda **solo** del catálogo candidato, con una frase de justificación en español cada una.
3. Probad con 1–2 géneros (p. ej. `["Action", "Sci-Fi"]`): mostrad candidatas y respuesta del modelo.

In [38]:
# --- Parte 4A (opcional): pitch_es, edad_sugerida, temas ---
# Requiere: `movies10` con `overview`; `client` y `MODEL` de la Parte 3.

In [39]:
# --- Parte 4B (opcional): recomendador por género (SOLUCIÓN) ---
# Requiere: `ratings_movies` (Parte 1); `client` y `MODEL` (Parte 3).